In [9]:
import requests
import pickle as pkl
from tqdm import tqdm
import pandas as pd
import sys
import time
import os
from random import choices
from numba import jit, cuda
from gpuparallel import GPUParallel, delayed
import itertools

In [10]:
with open('../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl', 'rb') as f:
    prop_typ_id_dict= pkl.load(f)

In [3]:
pos_ids = []
neg_ids = []
for study in prop_typ_id_dict.keys():
    study_dict = prop_typ_id_dict[study]
    train_len = len(study_dict['pos']['train'])
#     print('pos len', train_len)
    for pos_neg in study_dict.keys():
        pos_neg_dict = study_dict[pos_neg]
        
        for split in pos_neg_dict:
            
            if pos_neg == 'neg':
                if split == 'train':
                    sample = choices(pos_neg_dict[split], k=train_len)
                    neg_ids.extend(sample)
                else:
#                     print('neg len',split, len(pos_neg_dict[split]))
                    neg_ids.extend(pos_neg_dict[split])
                    
            else:
                pos_ids.extend(pos_neg_dict[split])
pos_ids = set(pos_ids)
neg_ids = set(neg_ids) - pos_ids
pos_ids = list(pos_ids)
neg_ids = list(neg_ids)

In [4]:
len(neg_ids)

2423359

In [5]:
len(pos_ids)

794795

In [6]:
filename_neg_human = '../data/pubmed_articles_neg_human.csv'
pmids = []

chunksize = 10**4
for chunk in tqdm(pd.read_csv(filename_neg_human, chunksize=chunksize)):
    row_ids = list(set(neg_ids).intersection(set(chunk.index)))
    pmids.extend(chunk['pmid'])
# neg_data_human = pd.read_csv(
#          filename_neg_human,
# )

30it [00:09,  3.28it/s]


KeyboardInterrupt: 

In [11]:
with open('neg_pmids.pkl', 'wb') as f:
    pkl.dump(pmids, f)

In [24]:
# pmids = list(pmids.iloc[neg_ids])

In [13]:
# pmids = list(neg_data_human.iloc[neg_ids]['pmid'])

In [ ]:
# for proc in range(20):
#     print(proc*len(pmids)//20, (proc+1)*len(pmids)//20)

In [ ]:
# proc_number = 2
# num_processes = 20
# for pmid in pmids[proc_number*len(pmids)//num_processes:(proc_number+1)*len(pmids)//num_processes]:
#     print(pmid)

In [ ]:
# import numpy as np
# @jit(target_backend='cuda')                         
# def func2(a):
#     for i in range(10000000):
#         a[i]+= 1
#         print(a[i])


In [11]:
            pmid = 100
            open_alex_pmid = f"open_alex_citations/open_alex_pubmed_pmid{pmid}.json"
            
            if not os.path.exists(open_alex_pmid):
                success = False
                retries = 1
                while not success:

                    response = requests.get(f"http://api.openalex.org/works/pmid:{pmid}")
                    if response.status_code == 200:
                            cited_by_url = response.json()['cited_by_api_url']


                            while not success:
                                response_cited_by = requests.get(cited_by_url)
                                if response_cited_by.status_code == 200:
                                    with open(open_alex_pmid, 'w') as f_pmid:
                                        json.dump(response_cited_by.json(), f_pmid)

                                    seen.append(pmid)
                                    success = True
                                    with open(f'open_alex_citations/pmids_seen{proc_number}.json', 'w') as f_seen:
                                        json.dump(seen, f_seen)
                                elif response.status_code == 429:
                                    wait_inner = retries_inner * 30;
                #                     wait = 60 * 60 * 24 + 10
                                    print(f'Error! Waiting %s secs and re-trying...{wait}')
                                    sys.stdout.flush()
                                    time.sleep(wait_inner)
                                    retries_inner += 1
                    elif response.status_code == 429:
                        print(response.status_code)
                        wait = retries * 30;
    #                     wait = 60 * 60 * 24 + 10
                        print(response.content)
                        print(f'Error! Waiting %s secs and re-trying...{wait}')
                        sys.stdout.flush()
                        time.sleep(wait)
                        retries += 1

In [17]:
import json
json.dumps([res['id'] for res in response_cited_by.json()['results'] ])

'["https://openalex.org/W2407551923", "https://openalex.org/W3030802447", "https://openalex.org/W2133316483", "https://openalex.org/W2414271397"]'

In [10]:
# @jit(target_backend='cuda') 
# def search_for_files(proc_number, device_id):
def search_for_files(pmid, worker_id, gpu_id):
            seen = []

#     for pmid in tqdm(pmids[proc_number*len(pmids)//num_processes:(proc_number+1)*len(pmids)//num_processes]):
#     for pmid in tqdm(pmids):
            open_alex_pmid = f"open_alex_citations/open_alex_pubmed_pmid{pmid}.json"
            
            if not os.path.exists(open_alex_pmid):
                success = False
                
                retries = 1
                while not success:

                    response = requests.get(f"http://api.openalex.org/works/pmid:{pmid}")
                    if response.status_code == 200:
                            cited_by_url = response.json()['cited_by_api_url']


                            while not success:
                                response_cited_by = requests.get(cited_by_url)
                                if response_cited_by.status_code == 200:
                                    with open(open_alex_pmid, 'w') as f_pmid:
                                        json.dump(response_cited_by.json(), f_pmid)

                                    seen.append(pmid)
                                    success = True
                                    with open(f'open_alex_citations/pmids_seen{proc_number}.json', 'w') as f_seen:
                                        json.dump(seen, f_seen)
                                elif response.status_code == 429:
                                    wait_inner = retries_inner * 30;
                #                     wait = 60 * 60 * 24 + 10
                                    print(f'Error! Waiting %s secs and re-trying...{wait}')
                                    sys.stdout.flush()
                                    time.sleep(wait_inner)
                                    retries_inner += 1
                    elif response.status_code == 429:
                        print(response.status_code)
                        wait = retries * 30;
    #                     wait = 60 * 60 * 24 + 10
                        print(response.content)
                        print(f'Error! Waiting %s secs and re-trying...{wait}')
                        sys.stdout.flush()
                        time.sleep(wait)
                        retries += 1
                    else:
                        break

In [8]:
# search_for_files(pmids)
pmids

[21054767,
 21054769,
 21054770,
 21054773,
 21054778,
 21054800,
 31006728,
 31006729,
 12074203,
 12074206,
 31006730,
 31006732,
 31006731,
 8840505,
 12074223,
 12074224,
 31006806,
 18222648,
 31006815,
 12074226,
 31006818,
 18222671,
 12074235,
 12074234,
 18222683,
 18222667,
 31006833,
 31006835,
 18222681,
 21054971,
 12074236,
 12074237,
 31006875,
 21054990,
 12074253,
 12074254,
 18222719,
 12074256,
 21054991,
 21055014,
 21055021,
 18222735,
 8840615,
 12074271,
 31006930,
 31006941,
 18222742,
 12074277,
 31006955,
 31006961,
 31006976,
 31006978,
 8840633,
 18222951,
 31006989,
 31007000,
 18222957,
 21055087,
 21055090,
 21055092,
 18222956,
 31007049,
 18222976,
 18222988,
 18223010,
 18223018,
 8840712,
 31007103,
 18223031,
 21055113,
 12074345,
 31007114,
 21055127,
 8840744,
 12074357,
 31007160,
 12074359,
 21055140,
 21055142,
 8840756,
 21055143,
 12074370,
 12074358,
 31007170,
 12074361,
 12074373,
 31007213,
 31007215,
 12074395,
 12074399,
 31007238,
 1207

In [14]:
gp = GPUParallel(n_gpu=2, n_workers_per_gpu=2)
results = gp(delayed(search_for_files)(pmid) for pmid in pmids)

Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, gpu_id=gpu_id)
  File "/tmp/ipykernel_926762/2391941193.py", line 24, in search_for_files
    json.dump(response_cited_by.json(), f_pmid)
NameError: name 'json' is not defined

Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, gpu_id=gpu_id)
  File "/tmp/ipykernel_926762/2391941193.py", line 24, in search_for_files
    json.dump(response_cited_by.json(), f_pmid)
NameError: name 'json' is not defined

Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, g

429
b'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 3.2 Final//EN">\n<title>429 Too Many Requests</title>\n<h1>Too Many Requests</h1>\n<p>10 per 1 second</p>\n'
Error! Waiting %s secs and re-trying...30


Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, gpu_id=gpu_id)
  File "/tmp/ipykernel_926762/2391941193.py", line 24, in search_for_files
    json.dump(response_cited_by.json(), f_pmid)
NameError: name 'json' is not defined

Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, gpu_id=gpu_id)
  File "/tmp/ipykernel_926762/2391941193.py", line 24, in search_for_files
    json.dump(response_cited_by.json(), f_pmid)
NameError: name 'json' is not defined

Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, g

Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, gpu_id=gpu_id)
  File "/tmp/ipykernel_926762/2391941193.py", line 24, in search_for_files
    json.dump(response_cited_by.json(), f_pmid)
NameError: name 'json' is not defined

Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, gpu_id=gpu_id)
  File "/tmp/ipykernel_926762/2391941193.py", line 24, in search_for_files
    json.dump(response_cited_by.json(), f_pmid)
NameError: name 'json' is not defined

Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, g

KeyboardInterrupt: 

Process ForkPoolWorker-11:
Process ForkPoolWorker-13:
Process ForkPoolWorker-12:
Traceback (most recent call last):
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/gpuparallel/gpuparallel.py", line 28, in _run_task
    result = func(worker_id=worker_id, gpu_id=gpu_id)
  File "/tmp/ipykernel_926762/2391941193.py", line 44, in search_for_files
    time.sleep(wait)
KeyboardInterrupt
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call la

  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/urllib3/util/ssl_.py", line 449, in ssl_wrap_socket
    ssl_sock = _ssl_wrap_socket_impl(
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/http/client.py", line 316, in begin
    version, status, reason = self._read_status()
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/urllib3/util/ssl_.py", line 493, in _ssl_wrap_socket_impl
    return ssl_context.wrap_socket(sock, server_hostname=server_hostname)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/ssl.py", line 1241, in recv_into
    return self.read(nbytes, buffer)
  File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/http/client.py", line 277, in _read_status
    line = str(self.fp.rea

In [ ]:
# open_alex_pmids = []
# not_found = []
# # open_alex_path = 

# for pmid in tqdm(pmids):
#     success = False
#     retries = 1
#     open_alex_pmid = f"open_alex_citations/open_alex_pubmed_pmid{pmid}.json"
#     if not os.path.exists(open_alex_pmid):
#         while not success:

#             response = requests.get(f"http://api.openalex.org/works/pmid:{pmid}")
#             if response.status_code == 200:
#                     open_alex_pmids.append(response.json())
#             elif response.status_code == 429:
#                 print(response.status_code)
#                 wait = retries * 30;
#                 print(f'Error! Waiting %s secs and re-trying...{wait}')
#                 sys.stdout.flush()
#                 time.sleep(wait)
#                 retries += 1
#             else:
#                 break

In [55]:
not_found

[0]

In [50]:
pmids.sort()

In [6]:
help(response)

Help on Response in module requests.models object:

class Response(builtins.object)
 |  The :class:`Response <Response>` object, which contains a
 |  server's response to an HTTP request.
 |  
 |  Methods defined here:
 |  
 |  __bool__(self)
 |      Returns True if :attr:`status_code` is less than 400.
 |      
 |      This attribute checks if the status code of the response is between
 |      400 and 600 to see if there was a client error or a server error. If
 |      the status code, is between 200 and 400, this will return True. This
 |      is **not** a check to see if the response code is ``200 OK``.
 |  
 |  __enter__(self)
 |  
 |  __exit__(self, *args)
 |  
 |  __getstate__(self)
 |  
 |  __init__(self)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __iter__(self)
 |      Allows you to use a response as an iterator.
 |  
 |  __nonzero__(self)
 |      Returns True if :attr:`status_code` is less than 400.
 |      
 |      This attribute checks if

In [34]:
response.text

'{"id":"https://openalex.org/W1992894650","doi":"https://doi.org/10.1016/0006-291x(75)90482-9","title":"Delineation of the intimate details of the backbone conformation of pyridine nucleotide coenzymes in aqueous solution","display_name":"Delineation of the intimate details of the backbone conformation of pyridine nucleotide coenzymes in aqueous solution","publication_year":1975,"publication_date":"1975-10-27","ids":{"openalex":"https://openalex.org/W1992894650","doi":"https://doi.org/10.1016/0006-291x(75)90482-9","mag":"1992894650","pmid":"https://pubmed.ncbi.nlm.nih.gov/2"},"host_venue":{"id":"https://openalex.org/V151589183","issn_l":"0006-291X","issn":["0006-291X","1090-2104"],"display_name":"Biochemical and Biophysical Research Communications","publisher":"Elsevier","type":"journal","url":"https://doi.org/10.1016/0006-291x(75)90482-9","is_oa":false,"version":null,"license":null},"type":"journal-article","open_access":{"is_oa":false,"oa_status":"closed","oa_url":null},"authorships"

In [35]:
response = requests.get(f"http://api.openalex.org/works/pmid:{2}")

In [37]:
response.text

'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 3.2 Final//EN">\n<title>429 Too Many Requests</title>\n<h1>Too Many Requests</h1>\n<p>500000 per 1 day</p>\n'

In [11]:
open_alex_pmids[0].keys()

dict_keys(['id', 'doi', 'title', 'display_name', 'publication_year', 'publication_date', 'ids', 'host_venue', 'type', 'open_access', 'authorships', 'cited_by_count', 'biblio', 'is_retracted', 'is_paratext', 'concepts', 'mesh', 'alternate_host_venues', 'referenced_works', 'related_works', 'ngrams_url', 'abstract_inverted_index', 'cited_by_api_url', 'counts_by_year', 'updated_date', 'created_date'])

In [14]:
response_cited_by = requests.get(open_alex_pmids[0]['cited_by_api_url'])

In [23]:
response_cited_by.json()['results'][1]['id']

'https://openalex.org/W2010878214'

In [24]:
open_alex_pmids[0]['cited_by_api_url']

'https://api.openalex.org/works?filter=cites:W1992894650'

In [33]:
import os
open_alex_pmid = f"open_alex_citations/open_alex_pubmed_pmid{2}.json"
os.path.exists(open_alex_pmid)

True